# 08-5. 로컬 HTTP 점검기 도구화 프로젝트 예제

## Goal

- 설정 검증과 실행 계획을 분리합니다.
- dry-run에서 네트워크·파일 쓰기를 수행하지 않습니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

합성 설정만 사용하며 HTTP 요청과 결과 파일 저장은 수행하지 않습니다.


## Steps

### 검증된 공개 실행 계획 만들기

비밀값을 제외한 설정만 dry-run 결과로 공개합니다.


In [1]:
import ipaddress
from urllib.parse import urlsplit


def validate_target(url: str) -> str:
    parts = urlsplit(url)
    if parts.scheme != "http" or not parts.hostname:
        raise ValueError("HTTP 기준 URL이 필요합니다")
    host = parts.hostname.lower()
    if host != "localhost" and not ipaddress.ip_address(host).is_loopback:
        raise ValueError("루프백 대상만 허용합니다")
    if parts.username or parts.password or parts.query or parts.fragment:
        raise ValueError("사용자정보·쿼리·프래그먼트는 허용하지 않습니다")
    return f"http://{host}:{parts.port or 80}"


def dry_run_plan(settings):
    return {
        "mode": "dry-run",
        "target": validate_target(settings["target"]),
        "output": settings["output"],
        "timeout": float(settings["timeout"]),
        "will_request": False,
        "will_write": False,
    }


plan = dry_run_plan({"target": "http://127.0.0.1:8080", "output": "artifacts/report.json", "timeout": 3})
print(plan)


{'mode': 'dry-run', 'target': 'http://127.0.0.1:8080', 'output': 'artifacts/report.json', 'timeout': 3.0, 'will_request': False, 'will_write': False}


## Checks

dry-run의 부작용 없음과 대상 범위를 확인합니다.


In [2]:
assert plan["will_request"] is False and plan["will_write"] is False
assert plan["target"] == "http://127.0.0.1:8080"
assert set(plan) == {"mode", "target", "output", "timeout", "will_request", "will_write"}
print("도구화 계획 검사 통과")


도구화 계획 검사 통과


## Next Steps

실제 CLI 계약은 `examples/08-toolization-project/test_local_http_tool.py`로 검증합니다.
